In [ ]:
# ==============================
# IMPORTS
# ==============================
import cv2
import winsound
import os
import threading

# ==============================
# LOAD HAAR CASCADE
# ==============================
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 
                                     'haarcascade_frontalface_default.xml')

eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 
                                    'haarcascade_eye.xml')

# ==============================
# AUDIO SETUP (REAL SOUND)
# ==============================
alarm_path = os.path.join(os.getcwd(), "alarm.wav")

def play_alarm():
    os.system(f'start {alarm_path}')  # frequency, duration

# ==============================
# VARIABLES
# ==============================
eye_closed_frames = 0
threshold = 15
alarm_on = False

# ==============================
# START WEBCAM
# ==============================
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ Cannot open webcam")
    exit()

print("🚀 Running Drowsiness Detection (Press 'q' to quit)")

# ==============================
# MAIN LOOP
# ==============================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]

        eyes = eye_cascade.detectMultiScale(roi_gray)

        # If no eyes detected → possible drowsiness
        if len(eyes) == 0:
            eye_closed_frames += 1
        else:
            eye_closed_frames = 0
            alarm_on = False

        # ALERT TRIGGER
        if eye_closed_frames > threshold:
            cv2.putText(frame, "DROWSINESS ALERT!", (100, 300),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

            if not alarm_on:
                alarm_on = True
                threading.Thread(target=play_alarm, daemon=True).start()

        # Draw eye boxes
        for (ex, ey, ew, eh) in eyes:
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (0,255,0), 2)

    cv2.imshow("Driver Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ==============================
# CLEANUP
# ==============================
cap.release()
cv2.destroyAllWindows()